# Clase 221 — Cold-start: popularity Bayesiana, onboarding, bandits

Estrategias para users/items sin historia. Demos sintéticos.

In [ ]:
import numpy as np, pandas as pd

# Items con n_ratings y mean_rating distintos
items = pd.DataFrame({
    'item': range(1, 11),
    'n':    [2, 5, 10, 50, 100, 500, 1000, 1500, 3, 1],
    'mean': [5.0, 4.8, 4.5, 4.4, 4.3, 4.2, 4.1, 4.0, 5.0, 5.0],
})
items['naive_score'] = items['mean']

# Popularity vanilla: ordenar por mean → ganan items con 1-2 ratings 5/5 (overconfident)
print('--- Popularity vanilla (por mean) ---')
print(items.sort_values('naive_score', ascending=False).head(5).to_string(index=False))

## 1. Bayesian shrinkage

In [ ]:
C = (items['mean'] * items['n']).sum() / items['n'].sum()
m = 50   # prior weight: equivale a 50 ratings al promedio global
items['bayes_score'] = (items['n'] * items['mean'] + m * C) / (items['n'] + m)

print(f'global mean C = {C:.3f}\n')
print('--- Bayesian shrinkage ---')
print(items.sort_values('bayes_score', ascending=False).head(5).to_string(index=False))
print('\n→ Items con n bajo (2, 1, 3) caen porque el prior los ancla a C.')
print('  Items con n alto y rating bueno (1000-1500) dominan.')

## 2. Onboarding: usuario nuevo elige géneros

In [ ]:
rng = np.random.default_rng(42)
n_items = 200
GENRES = ['action', 'comedy', 'drama', 'sci-fi', 'romance']
item_genres = pd.DataFrame({
    'item': range(n_items),
    'genre': rng.choice(GENRES, size=n_items),
    'popularity': rng.integers(10, 5000, n_items),
    'mean_rating': np.clip(rng.normal(3.8, 0.4, n_items), 1, 5),
})

def onboarding_recommendations(selected_genres, df, n=10):
    """User nuevo elige géneros → recomendar mejores items en esos géneros con Bayesian shrinkage."""
    df = df[df.genre.isin(selected_genres)].copy()
    C = df['mean_rating'].mean()
    m = 100
    df['score'] = (df['popularity'] * df['mean_rating'] + m * C) / (df['popularity'] + m)
    return df.nlargest(n, 'score')[['item', 'genre', 'popularity', 'mean_rating', 'score']]

print('User nuevo eligió: sci-fi, action')
print(onboarding_recommendations(['sci-fi', 'action'], item_genres).to_string(index=False))

## 3. Item cold-start con content-based

In [ ]:
# Agregamos 3 movies nuevas (0 ratings)
new_movies = pd.DataFrame([
    {'item': 999, 'genre': 'sci-fi', 'popularity': 0, 'mean_rating': np.nan},
    {'item': 998, 'genre': 'action', 'popularity': 0, 'mean_rating': np.nan},
    {'item': 997, 'genre': 'comedy', 'popularity': 0, 'mean_rating': np.nan},
])
all_items = pd.concat([item_genres, new_movies], ignore_index=True)

def cold_start_recs(selected_genres, df, n=10, novelty_boost=True):
    # Items sin historia: usar prior global como rating; boost por ser nuevo
    df = df.copy()
    C_global = df['mean_rating'].mean()
    is_new = df['mean_rating'].isna()
    df.loc[is_new, 'mean_rating'] = C_global
    df.loc[is_new, 'popularity'] = 100   # "weight" pequeño
    df['is_new'] = is_new

    df = df[df.genre.isin(selected_genres)]
    df['score'] = df['mean_rating'].copy()
    if novelty_boost:
        df.loc[df.is_new, 'score'] = df.loc[df.is_new, 'score'] + 0.5   # boost a items nuevos
    return df.nlargest(n, 'score')[['item', 'genre', 'popularity', 'mean_rating', 'is_new', 'score']]

print('Cold-start de items nuevos en sci-fi/action:')
print(cold_start_recs(['sci-fi', 'action'], all_items).to_string(index=False))
print('\n→ items_999 y 998 (nuevos) están en el top por el novelty boost.')

## 4. Epsilon-greedy bandit

In [ ]:
class EpsilonGreedy:
    def __init__(self, n_arms, eps=0.1, rng=None):
        self.n_arms = n_arms
        self.eps = eps
        self.rewards = np.zeros(n_arms)
        self.pulls = np.zeros(n_arms, dtype=int)
        self.rng = rng or np.random.default_rng(0)

    def select(self):
        if self.rng.random() < self.eps:
            return self.rng.integers(self.n_arms)
        avg = np.where(self.pulls > 0, self.rewards / np.maximum(self.pulls, 1), 0)
        return int(np.argmax(avg))

    def update(self, arm, reward):
        self.pulls[arm] += 1
        self.rewards[arm] += reward

# Simular 1000 visitas con 5 "items" cuya conversion real es distinta
true_conv = np.array([0.10, 0.05, 0.20, 0.08, 0.15])
bandit = EpsilonGreedy(n_arms=5, eps=0.1, rng=np.random.default_rng(1))

for _ in range(1000):
    arm = bandit.select()
    reward = int(np.random.random() < true_conv[arm])
    bandit.update(arm, reward)

print(f'{"arm":>4} {"true_conv":>10} {"pulls":>7} {"observed":>10}')
for a in range(5):
    obs = bandit.rewards[a] / max(bandit.pulls[a], 1)
    print(f'{a:>4} {true_conv[a]:>10.3f} {bandit.pulls[a]:>7d} {obs:>10.3f}')
print(f'\n→ arm 2 (true_conv 0.20) recibió la mayoría de pulls.')
print(f'  arm 1 (true_conv 0.05) recibió pocos — eps explora pero no estanca ahí.')

## Ejercicio guiado

1. Sobre MovieLens, simulá 100 users nuevos (sin historia). Compará NDCG@10 con: (a) popularity vanilla, (b) Bayesian shrinkage, (c) onboarding + content-based.
2. Para items nuevos: trackeá tiempo hasta acumular 10 ratings comparando estrategia "sin boost" vs "con novelty boost".
3. Thompson sampling vs epsilon-greedy: implementá Thompson sampling con priors Beta para conversiones binarias.
4. Contextual bandit: usa demographics del user como contexto (edad, país). Compará vs vanilla bandit.
5. Bonus: cross-domain transfer: importa "gustos" de un dataset (música) y prediga preferencias en otro (películas).

## Conclusiones

- Cold-start es PRODUCTO, no solo modelo: onboarding, novelty boost, fallbacks son decisiones de UX.
- Bayesian shrinkage es la fix más barata y efectiva para popularity ranking.
- Content features (Clase 218) son la llave para item cold-start.
- Bandits resuelven la exploración cuando offline metrics no alcanzan.